# Face Recognition Project

Axel Omar Sanchez Peralta (UCID: 30145429)

Mariia Podgaietska (UCID: )

## 1. Introduction

## 2. Get Setup

### 2.1 Download Dataset Helper Function

In [ ]:
import os
import zipfile
from pathlib import Path
import requests

def download_data(source: str, 
                    destination: str,
                    remove_source: bool = True) -> Path:
    """Downloads a zipped dataset from source and unzips to destination.

    Args:
        source (str): A link to a zipped file containing data.
        destination (str): A target directory to unzip data to.
        remove_source (bool): Whether to remove the source after downloading and extracting.
    
    Returns:
        pathlib.Path to downloaded data.
    
    Example usage:
        download_data(source="https://github.com/mrdbourke/pytorch-deep-learning/raw/main/data/pizza_steak_sushi.zip",
        destination="pizza_steak_sushi")
    """
    # Setup path to data folder
    data_path = Path("data/")
    image_path = data_path / destination

    # If the image folder doesn't exist, download it and prepare it... 
    if image_path.is_dir():
        print(f"[INFO] {image_path} directory exists, skipping download.")
    else:
        print(f"[INFO] Did not find {image_path} directory, creating one...")
        image_path.mkdir(parents=True, exist_ok=True)
        
        # Download pizza, steak, sushi data
        target_file = Path(source).name
        print(f"[INFO] Downloading {target_file} from {source}...")
        with requests.get(source, stream=True, timeout=30) as response:
            response.raise_for_status()
            with open(data_path / target_file, "wb") as f:
                for chunk in response.iter_content(chunk_size=8192):
                    if chunk:
                        f.write(chunk)

        # Unzip pizza, steak, sushi data
        with zipfile.ZipFile(data_path / target_file, "r") as zip_ref:
            print(f"[INFO] Unzipping {target_file} data...") 
            zip_ref.extractall(image_path)

        # Remove .zip file
        if remove_source:
            os.remove(data_path / target_file)
    
    return image_path

### 2.2 Unzip Dataset

In [ ]:
from pathlib import Path
import zipfile

dataset_dir = Path("data/dataset")
zip_path = Path("data/dataset.zip")

if dataset_dir.exists():
    print("[INFO] Dataset already available.")
elif zip_path.exists():
    print("[INFO] Unzipping dataset...")
    dataset_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(dataset_dir)
else:
    print("[INFO] Dataset zip not found locally... downloading dataset...")
    download_data(source="https://github.com/Axeloooo/Face-Recognition/raw/devel/data/dataset.zip",
                    destination="dataset")

## 3. Get Data

In [ ]:
import cv2
import numpy as np
from glob import glob
from pathlib import Path

# custom load data function to load images and record subject labels
def get_data(path: str):
    """Loads images from a given path, extracts subject labels from the directory structure, and preprocesses the images.

    Args:
        path (str): A glob pattern to match image files (e.g., 'data/dataset/s*/**/*.jpg').

    Returns:
        Tuple[np.ndarray, np.ndarray]: A tuple containing the preprocessed image data and corresponding labels.
    """
    paths = glob(path, recursive=True)
    data = [] #list of images
    label = [] #list of labels

    for path in paths:
        img = cv2.imread(path,0) # read image
        if img is None:
            continue
        # Extract subject label from path using pathlib (OS-independent)
        path_parts = Path(path).parts
        subject_folder = [part for part in path_parts if part.startswith('s') and part[1:].isdigit()]
        if subject_folder:
            subject_label = subject_folder[0][1:]  # Remove 's' prefix
        else:
            continue  # Skip if subject folder not found

        # pre-processing step
        # can resize, rescale, normalize
        img = img.reshape(-1) # reshape image to a 1D vector
        img = np.float32(img / 255.0) #normalize to 0-1 value

        # can apply LBP, PCA or other forms of feature extraction
        # append images and labels
        data.append(img)

        # decrease all labels by 1 since subject labels start from 1
        label.append(int(subject_label)-1)
        
    return np.array(data), np.array(label)

## 4. Local Binary Pattern (LBP) 

In [ ]:
from skimage.feature import local_binary_pattern
import cv2
import numpy as np
from glob import glob
from pathlib import Path

def get_data_lbp(path: str, n_bins: int = 64):
    """Load images and extract LBP histogram feature vectors.

    Each image is converted to a normalised histogram of Local Binary Pattern
    values (P=12 neighbours, radius R=3, method='default').  The histogram
    acts as a compact, fixed-length texture descriptor of length n_bins.

    Args:
        path (str): Glob pattern pointing to .pgm image files.
        n_bins (int): Number of histogram bins (controls descriptor length). For P=12, LBP values span [0, 2^12 - 1] = [0, 4095].

    Returns:
        Tuple[np.ndarray, np.ndarray]: (data, labels) arrays.
    """
    paths = glob(path, recursive=True)
    data, label = [], []
    lbp_max = 2 ** 12  # value range upper bound for P=12

    for p in paths:
        img = cv2.imread(p, 0)
        if img is None:
            continue
        path_parts = Path(p).parts
        subject_folder = [part for part in path_parts if part.startswith('s') and part[1:].isdigit()]
        if not subject_folder:
            continue

        # Use the original integer grayscale image for LBP computation
        # to avoid sensitivity to tiny floating-point differences
        img_u8 = img.astype(np.uint8, copy=False)
        lbp = local_binary_pattern(img_u8, P=12, R=3, method='default')
        hist, _ = np.histogram(lbp.ravel(), bins=n_bins, range=(0, lbp_max), density=True)

        data.append(hist)
        label.append(int(subject_folder[0][1:]) - 1)

    return np.array(data), np.array(label)

## 5. Support Vector Machines (SVM)

![](https://github.com/Axeloooo/Face-Recognition/raw/devel/images/support-vector-machines.png)

In [ ]:
from sklearn.svm import SVC
from sklearn.model_selection import RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import numpy as np

LBP_N_BINS = 64  # histogram bins for LBP — mirrors the value used in Section 4

_base      = "data/dataset/ATT dataset/s*"
train_path = f"{_base}/[1-8].pgm"

# ── Load both feature representations ─────────────────────────────────────────
train_raw, train_y_raw   = get_data(train_path)
_t9r,  _l9r              = get_data(f"{_base}/9.pgm")
_t10r, _l10r             = get_data(f"{_base}/10.pgm")
test_raw,  test_y_raw    = np.concatenate([_t9r,  _t10r]), np.concatenate([_l9r,  _l10r])

train_lbp, train_y_lbp  = get_data_lbp(train_path, n_bins=LBP_N_BINS)
_t9l,  _l9l              = get_data_lbp(f"{_base}/9.pgm",  n_bins=LBP_N_BINS)
_t10l, _l10l             = get_data_lbp(f"{_base}/10.pgm", n_bins=LBP_N_BINS)
test_lbp,  test_y_lbp   = np.concatenate([_t9l,  _t10l]), np.concatenate([_l9l,  _l10l])

# ── Pipeline + parameter grid ──────────────────────────────────────────────────
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('svc', SVC(probability=True, random_state=42)),
])

# 4 kernels × 5 C values × 4 gamma values = 80 combos × 5 folds = 400 fits - too costly
# for exhaustive search. RandomizedSearchCV samples n_iter=30 random combos
# instead (~83 % cheaper) while still giving broad coverage of the space.
param_grid = {
    'svc__kernel': ['rbf', 'linear', 'poly', 'sigmoid'],
    'svc__C':      [0.1, 1.0, 5.0, 10.0, 100.0],
    'svc__gamma':  ['scale', 0.001, 0.01, 0.1],
}

# ── Train on both representations; store all results for Sections 7 & 8 ───────
model_results = {}  # shared dict — extended by Section 6, consumed by 7 & 8

for name, (train_X, train_y, test_X, test_y) in [
    ('svm_raw', (train_raw, train_y_raw, test_raw, test_y_raw)),
    ('svm_lbp', (train_lbp, train_y_lbp, test_lbp, test_y_lbp)),
]:
    search = RandomizedSearchCV(pipeline, param_grid, n_iter=20,cv=5, scoring='accuracy', n_jobs=-1, random_state=42)
    search.fit(train_X, train_y)
    prob = search.predict_proba(test_X)
    pred = np.argmax(prob, axis=1)
    acc  = np.mean(pred == test_y)
    model_results[name] = {
        'probability_matrix': prob,
        'prediction':         pred,
        'test_label':         test_y,
        'accuracy':           acc,
    }
    print(f"[{name.upper():<7}] Best params  : {search.best_params_}")
    print(f"[{name.upper():<7}] CV accuracy  : {search.best_score_:.4f}")
    print(f"[{name.upper():<7}] Test accuracy: {acc:.4f}\n")

## 6. Multi-Layer Perceptron (MLP)

![](https://github.com/Axeloooo/Face-Recognition/raw/devel/images/multi-layer-perceptron.png)

In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import numpy as np

LBP_N_BINS = 64  # must match the value used in Section 5

_base      = "data/dataset/ATT dataset/s*"
train_path = f"{_base}/[1-8].pgm"

# ── Load both feature representations ─────────────────────────────────────────
train_raw, train_y_raw   = get_data(train_path)
_t9r,  _l9r              = get_data(f"{_base}/9.pgm")
_t10r, _l10r             = get_data(f"{_base}/10.pgm")
test_raw,  test_y_raw    = np.concatenate([_t9r,  _t10r]), np.concatenate([_l9r,  _l10r])

train_lbp, train_y_lbp  = get_data_lbp(train_path, n_bins=LBP_N_BINS)
_t9l,  _l9l              = get_data_lbp(f"{_base}/9.pgm",  n_bins=LBP_N_BINS)
_t10l, _l10l             = get_data_lbp(f"{_base}/10.pgm", n_bins=LBP_N_BINS)
test_lbp,  test_y_lbp   = np.concatenate([_t9l,  _t10l]), np.concatenate([_l9l,  _l10l])

# ── Pipeline + parameter distribution ─────────────────────────────────────────
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('mlp', MLPClassifier(max_iter=500, random_state=42)),
])

# Full grid: 5 × 2 × 2 × 3 × 3 = 180 combos × 5 folds = 900 fits — too costly
# for exhaustive search.  RandomizedSearchCV samples n_iter=30 random combos
# instead (~83 % cheaper) while still giving broad coverage of the space.
param_dist = {
    'mlp__hidden_layer_sizes': [(64,), (128,), (64, 64), (128, 64), (128, 64, 128)],
    'mlp__activation':         ['relu', 'tanh'],
    'mlp__solver':             ['adam', 'sgd'],
    'mlp__learning_rate_init': [0.001, 0.01, 0.1],
    'mlp__alpha':              [0.0001, 0.001, 0.01],
}

# ── Train on both representations; extend shared model_results ────────────────
for name, (train_X, train_y, test_X, test_y) in [
    ('mlp_raw', (train_raw, train_y_raw, test_raw, test_y_raw)),
    ('mlp_lbp', (train_lbp, train_y_lbp, test_lbp, test_y_lbp)),
]:
    search = RandomizedSearchCV(pipeline, param_dist, n_iter=30, cv=5, scoring='accuracy', n_jobs=-1, random_state=42)
    search.fit(train_X, train_y)
    prob = search.predict_proba(test_X)
    pred = np.argmax(prob, axis=1)
    acc  = np.mean(pred == test_y)
    model_results[name] = {
        'probability_matrix': prob,
        'prediction':         pred,
        'test_label':         test_y,
        'accuracy':           acc,
    }
    print(f"[{name.upper():<7}] Best params  : {search.best_params_}")
    print(f"[{name.upper():<7}] CV accuracy  : {search.best_score_:.4f}")
    print(f"[{name.upper():<7}] Test accuracy: {acc:.4f}\n")

print("model_results keys:", list(model_results.keys()))
print("All four configurations ready for Sections 7 & 8.")

## 7. Results

### 7.1 ROC (FPR vs. TPR)

### 7.2. DET (FPR vs. FNR)

## 9. Conclusion